Goal: 
Preprocess the data with supervised learning technique

Approach: 
Combines entity recognition (NER) preprocessing with a supervised relation extraction model (BertRelationExtractor), trained on a small dataset of sentences, entities, and labeled relations.

In [14]:
import torch
import transformers
from transformers import BertTokenizerFast, BertForTokenClassification, BertModel
from TorchCRF import CRF
from torch import nn
from torch.utils.data import DataLoader, Dataset
import numpy as np

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, f1_score

In [23]:
# train_entities = [("entity", "entity_type", idx)]
# train_relations = [(idx1, idx2, "relation_type)]
# introducing metadata

label_map = {
    "ENTITY": 0, 
    "YEAR": 1, 
    "VALUE": 2,
    "ENTITY_DESCRIPTOR": 3,
    "YEAR_DESCRIPTOR": 4
}

relation_map = {
    "no_relation": 0,
    "has_value": 1,
    "in_year": 2,
    "describes_entity": 3,
    "describes_year": 4
}


Assume the dataset is normalized beforehand

In [16]:
# train_entities = ("phrase:, "entity", "start_idx", "end_idx")

In [17]:
train_sentences = [
    "Our overall gross profit of our household cleaning tools increased from approximately 44.4 million dollars for the year ended 31 December 2018 to approximately 45.1 million dollars for the year ended 31 December 2019 , which was mainly due to the increase in gross profit of toilet cleaning tools",
    "Our overall gross profit of our household cleaning tools further increased to approximately 56.6 million dollars for the year ended 31 December 2020 , which was mainly due to the increase in gross profit of floor cleaning tools and toilet cleaning tools",
    "Our bank interest income amounted to approximately 0.4 million dollars , 0.7 million dollars , 0.7 million dollars and 0.2 million dollars for the years ended 31 December 2018 , 2019 , 2020 and the four months ended 30 April 2021 , respectively"
]

train_entities = [
    [
        ("gross profit", "ENTITY", 2, 3),
        ("household cleaning tools", "ENTITY_DESCRIPTOR", 6, 8),
        ("44.4 million dollars", "VALUE", 12, 14),
        ("year ended 31 December", "YEAR_DESCRIPTOR", 17, 20),
        ("2018", "YEAR", 21, 21),
        ("45.1 million dollars", "VALUE", 24, 26),
        ("year ended 31 December", "YEAR_DESCRIPTOR", 29, 32),
        ("2019", "YEAR", 33, 33)
    ],
    [
        ("gross profit", "ENTITY", 2, 3),
        ("household cleaning tools", "ENTITY_DESCRIPTOR", 6, 8),
        ("56.6 million dollars", "VALUE", 13, 15),
        ("year ended 31 December", "YEAR_DESCRIPTOR", 18, 21),
        ("2020", "YEAR", 22, 22)
    ],
    [
        ("bank interest income", "ENTITY", 1, 3),
        ("0.4 million dollars", "VALUE", 7, 9),
        ("0.7 million dollars", "VALUE", 11, 13),
        ("0.7 million dollars", "VALUE", 15, 17),
        ("0.2 million dollars", "VALUE", 19, 21),
        ("years ended 31 December", "YEAR_DESCRIPTOR", 24, 27),
        ("2018", "YEAR", 28, 28),
        ("2019", "YEAR", 30, 30),
        ("2020", "YEAR", 32, 32),
        ("four months ended 30 April", "YEAR_DESCRIPTOR", 35, 39),
        ("four months ended 30 April 2021", "YEAR", 35, 40)
    ]
]

train_relations = [
    [
        ((6, 8), (2, 3), "describes_entity"),
        ((2, 3), (12, 14), "has_value"),
        ((12, 14), (21, 21), "in_year"),
        ((17, 20), (21, 21), "describes_year"),
        ((2, 3), (24, 26), "has_value"),
        ((24, 26), (33, 33), "in_year"),
        ((29, 32), (33, 33), "describes_year"),
        ((2, 3), (33, 33), "no_relation")  # Negative example: gross profit -> 2019
    ],
    [
        ((6, 8), (2, 3), "describes_entity"),
        ((2, 3), (13, 15), "has_value"),
        ((13, 15), (22, 22), "in_year"),
        ((18, 21), (22, 22), "describes_year"),
        ((2, 3), (22, 22), "no_relation")  # Negative example: gross profit -> 2020
    ],
    [
        ((1, 3), (7, 9), "has_value"),
        ((1, 3), (11, 13), "has_value"),
        ((1, 3), (15, 17), "has_value"),
        ((1, 3), (19, 21), "has_value"),
        ((7, 9), (28, 28), "in_year"),
        ((11, 13), (30, 30), "in_year"),
        ((15, 17), (32, 32), "in_year"),
        ((19, 21), (35, 40), "in_year"),
        ((24, 27), (28, 28), "describes_year"),
        ((24, 27), (30, 30), "describes_year"),
        ((24, 27), (32, 32), "describes_year"),
        ((35, 39), (35, 40), "describes_year"),
        ((1, 3), (28, 28), "no_relation")  # Negative example: bank interest income -> 2018
    ]
]

In [18]:
class BertRelationExtractor(nn.Module):
    def __init__(self, num_labels=5):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(768 * 2, num_labels)

    def forward(self, input_ids, attention_mask, entity1_mask, entity2_mask, labels=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        sequence_output = outputs[0]
        e1_repr = (sequence_output * entity1_mask.unsqueeze(-1)).sum(dim=1) / entity1_mask.sum(dim=1).unsqueeze(-1)
        e2_repr = (sequence_output * entity2_mask.unsqueeze(-1)).sum(dim=1) / entity2_mask.sum(dim=1).unsqueeze(-1)
        combined = torch.cat((e1_repr, e2_repr), dim=-1)
        combined = self.dropout(combined)
        logits = self.classifier(combined)
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            return loss_fn(logits, labels)
        return logits

In [20]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
max_length = 100

In [25]:
def preprocess_relations(sentences, entities, relations):
    data = {"input_ids": [], "attention_mask": [], "entity1_mask": [], "entity2_mask": [], "labels": []}
    for sent, ents, rels in zip(sentences, entities, relations):
        # Tokenize with BERT
        encoding = tokenizer(sent, return_tensors="pt", padding="max_length", truncation=True, max_length=128)
        input_ids = encoding["input_ids"][0]
        attention_mask = encoding["attention_mask"][0]

        # Map space-split indices to BERT token indices
        space_tokens = sent.split()
        token_map = []
        bert_idx = 1  # Skip [CLS]
        for i, word in enumerate(space_tokens):
            word_tokens = tokenizer.tokenize(word)
            token_map.append(bert_idx)
            bert_idx += len(word_tokens)

        for (e1_start, e1_end), (e2_start, e2_end), rel in rels:
            e1_mask = torch.zeros_like(input_ids)
            e2_mask = torch.zeros_like(input_ids)

            # Map start/end indices to BERT
            e1_start_bert = token_map[e1_start]
            e1_end_bert = token_map[e1_end] + len(tokenizer.tokenize(space_tokens[e1_end])) - 1
            e2_start_bert = token_map[e2_start]
            e2_end_bert = token_map[e2_end] + len(tokenizer.tokenize(space_tokens[e2_end])) - 1

            e1_mask[e1_start_bert:e1_end_bert + 1] = 1
            e2_mask[e2_start_bert:e2_end_bert + 1] = 1

            data["input_ids"].append(input_ids)
            data["attention_mask"].append(attention_mask)
            data["entity1_mask"].append(e1_mask)
            data["entity2_mask"].append(e2_mask)
            data["labels"].append(relation_map[rel])

    # Convert to tensors
    for key in data:
        if key == "labels":
            data[key] = torch.tensor(data[key], dtype=torch.long)
        else:
            data[key] = torch.stack(data[key])
    return data

In [26]:
# Run preprocessing
data = preprocess_relations(train_sentences, train_entities, train_relations)

# Verify shapes
print("Data Shapes:")
for key, value in data.items():
    print(f"  {key}: {value.shape}")

Data Shapes:
  input_ids: torch.Size([26, 128])
  attention_mask: torch.Size([26, 128])
  entity1_mask: torch.Size([26, 128])
  entity2_mask: torch.Size([26, 128])
  labels: torch.Size([26])


In [28]:
# Helper function to inspect mappings
def inspect_mappings(sent, ents, rels):
    encoding = tokenizer(sent, return_tensors="pt", padding="max_length", truncation=True, max_length=128)
    input_ids = encoding["input_ids"][0]
    bert_tokens = tokenizer.convert_ids_to_tokens(input_ids)
    
    space_tokens = sent.split()
    token_map = []
    bert_idx = 1  # Skip [CLS]
    for i, word in enumerate(space_tokens):
        word_tokens = tokenizer.tokenize(word)
        token_map.append(bert_idx)
        bert_idx += len(word_tokens)
    
    print(f"Sentence: {sent}")
    print(f"Space Tokens: {space_tokens}")
    print(f"BERT Tokens: {bert_tokens[:len(bert_tokens)//2]}...")  # Truncate for brevity
    print("Entity Mappings:")
    for text, _, start, end in ents:
        start_bert = token_map[start]
        end_bert = token_map[end] + len(tokenizer.tokenize(space_tokens[end])) - 1
        bert_span = bert_tokens[start_bert:end_bert + 1]
        print(f"  {text} ({start}-{end}) -> BERT {start_bert}-{end_bert}: {bert_span}")
    
    print("Relation Mappings:")
    for (e1_start, e1_end), (e2_start, e2_end), rel in rels:
        e1_start_bert = token_map[e1_start]
        e1_end_bert = token_map[e1_end] + len(tokenizer.tokenize(space_tokens[e1_end])) - 1
        e2_start_bert = token_map[e2_start]
        e2_end_bert = token_map[e2_end] + len(tokenizer.tokenize(space_tokens[e2_end])) - 1
        e1_span = bert_tokens[e1_start_bert:e1_end_bert + 1]
        e2_span = bert_tokens[e2_start_bert:e2_end_bert + 1]
        print(f"  {rel}: ({e1_start}-{e1_end}) -> ({e2_start}-{e2_end}) -> BERT {e1_start_bert}-{e1_end_bert} -> {e2_start_bert}-{e2_end_bert}: {e1_span} -> {e2_span}")

In [29]:
# Inspect mappings for each sentence
for i, (sent, ents, rels) in enumerate(zip(train_sentences, train_entities, train_relations)):
    print(f"\n=== Sentence {i+1} ===")
    inspect_mappings(sent, ents, rels)


=== Sentence 1 ===
Sentence: Our overall gross profit of our household cleaning tools increased from approximately 44.4 million dollars for the year ended 31 December 2018 to approximately 45.1 million dollars for the year ended 31 December 2019 , which was mainly due to the increase in gross profit of toilet cleaning tools
Space Tokens: ['Our', 'overall', 'gross', 'profit', 'of', 'our', 'household', 'cleaning', 'tools', 'increased', 'from', 'approximately', '44.4', 'million', 'dollars', 'for', 'the', 'year', 'ended', '31', 'December', '2018', 'to', 'approximately', '45.1', 'million', 'dollars', 'for', 'the', 'year', 'ended', '31', 'December', '2019', ',', 'which', 'was', 'mainly', 'due', 'to', 'the', 'increase', 'in', 'gross', 'profit', 'of', 'toilet', 'cleaning', 'tools']
BERT Tokens: ['[CLS]', 'our', 'overall', 'gross', 'profit', 'of', 'our', 'household', 'cleaning', 'tools', 'increased', 'from', 'approximately', '44', '.', '4', 'million', 'dollars', 'for', 'the', 'year', 'ended', 

In [20]:
train_data = preprocess_relations(train_sentences, train_entities, train_relations)

In [ ]:
def split_data(data, val_split=0.2, random_seed=42):
    # Number of samples
    num_samples = len(data["input_ids"])
    indices = list(range(num_samples))
    
    # Split indices
    train_indices, val_indices = train_test_split(indices, test_size=val_split, random_state=random_seed)
    
    # Create train and val dictionaries
    train_split = {}
    val_split = {}
    for key in data:
        train_split[key] = data[key][train_indices]
        val_split[key] = data[key][val_indices]
    
    return train_split, val_split

In [ ]:
train_split, val_split = split_data(train_data, val_split=0.2)

In [21]:
class RelationDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data["input_ids"])

    def __getitem__(self, idx):
        return {key: self.data[key][idx] for key in self.data}

In [22]:
train_dataset = RelationDataset(train_split)
val_dataset = RelationDataset(val_split)

batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [13]:
print({k: v.shape for k, v in train_data.items()})

{'input_ids': torch.Size([30, 100]), 'attention_mask': torch.Size([30, 100]), 'entity1_mask': torch.Size([30, 100]), 'entity2_mask': torch.Size([30, 100]), 'labels': torch.Size([30])}


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
relation_map = {"no_relation": 0, "has_value": 1, "in_year": 2, "owned_by": 3}

model = BertRelationExtractor(num_labels=len(relation_map)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

In [22]:
def train_model(model, train_loader, val_loader, epochs=5, patience=3):
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    loss_fn = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            e1_mask = batch["entity1_mask"].to(device)
            e2_mask = batch["entity2_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            loss = model(input_ids, attention_mask, e1_mask, e2_mask, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)

        # Validation phase
        model.eval()
        val_loss = 0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                e1_mask = batch["entity1_mask"].to(device)
                e2_mask = batch["entity2_mask"].to(device)
                labels = batch["labels"].to(device)

                logits = model(input_ids, attention_mask, e1_mask, e2_mask)
                loss = loss_fn(logits, labels)
                val_loss += loss.item()

                preds = torch.argmax(logits, dim=-1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = accuracy_score(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds, average='weighted')  # Weighted F1 for multi-class
        
        print(f"Epoch {epoch+1}:")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss: {avg_val_loss:.4f}")
        print(f"  Val Accuracy: {val_accuracy:.4f}")
        print(f"  Val F1-Score: {val_f1:.4f}")

        #Early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict()  # Save best model weights
            patience_counter = 0
            print(f"  Validation loss improved to {best_val_loss:.4f}, saving model state.")
        else:
            patience_counter += 1
            print(f"  Validation loss did not improve. Patience counter: {patience_counter}/{patience}")
            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                break
    
    return model

In [23]:
train_model(model, train_loader, val_loader, epochs=5, patience=3)

Epoch 1, Loss: 1.4377
Epoch 2, Loss: 0.8303
Epoch 3, Loss: 0.7330
Epoch 4, Loss: 0.7583
Epoch 5, Loss: 0.7092
